# 02 — Building the analysis table

`01_qc.ipynb` decided **which** recordings are usable. This notebook decides **what shape** the data has to be in before anything is tested, and stops there.

Three sources have to meet: the sway parameters (one row per participant, 48 outcome columns), the LimeSurvey export (one row per participant, 100 mixed columns) and the VR folder names (one folder per session). Each carries the condition label in a different place, so most of the work here is making those labels agree before they are used as a grouping variable.

**Nothing in this notebook compares an outcome between conditions.** That separation is deliberate: the analysis is fixed in `ANALYSIS_PLAN.md`, which is committed before `03_analysis.ipynb` exists, so the git history shows the test was chosen without having seen its result.

*Ноутбук `01_qc.ipynb` определил, какие записи пригодны. Здесь решается, в какой форме данные должны оказаться до того, как что-либо будет протестировано — и на этом всё.*

*Сходятся три источника: параметры раскачивания (строка на участника, 48 колонок исходов), выгрузка LimeSurvey (строка на участника, 100 разнородных колонок) и имена папок VR (папка на сессию). Метка условия в каждом лежит в своём месте, поэтому основная работа — согласовать эти метки прежде, чем использовать их как группирующую переменную.*

***В этом ноутбуке нет ни одного сравнения исхода между условиями.*** *Разделение намеренное: анализ зафиксирован в `ANALYSIS_PLAN.md`, который коммитится до появления `03_analysis.ipynb`, — история git показывает, что тест выбран, не увидев своего результата.*

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

REPO = Path.cwd().parent
RAW = REPO / "data" / "raw"
DER = REPO / "data" / "derived"

# LimeSurvey writes 99 into a rating field the tester skipped; it is not a rating on the 0-10 scale.
MISSING_CODE = 99

# Participant codes as typed into LimeSurvey vs. as written in the file names.
# Fixed by hand against the participant list, not by fuzzy matching.
ID_FIXES = {"DO11ÜB30": "DO11UB30", "DI16ÜB31": "DI16UB31", "AN": "AN07IE11"}

BLOCKS = [1, 2, 3, 4]
SESSIONS = ["A", "B"]
CONDITIONS = ["HB", "LB"]

## 1. Sway parameters: wide to long

`balance_data_2026.csv` holds one row per participant and one column per *(parameter, condition, block)* combination — column names such as `PeriodicPower HB_t3`. Comparing conditions needs the opposite layout: one row per *(participant, condition, block)* and one column per parameter.

The reshape is done in two steps rather than one, so the intermediate is inspectable: `.melt()` collapses all 48 outcome columns into a single `col`/`value` pair, `.str.extract()` splits `col` into its three parts with a named regex, and `.pivot()` then spreads the parameter names back out as columns.

*В `balance_data_2026.csv` — строка на участника и колонка на каждое сочетание (параметр, условие, блок), с именами вида `PeriodicPower HB_t3`. Для сравнения условий нужна обратная раскладка: строка на (участник, условие, блок) и колонка на параметр.*

*Переформатирование делается в два шага, а не в один, чтобы промежуточный результат можно было посмотреть: `.melt()` схлопывает все 48 колонок исходов в пару `col`/`value`, `.str.extract()` разбирает `col` на три части именованным регулярным выражением, `.pivot()` возвращает имена параметров обратно в колонки.*

In [2]:
bal = pd.read_csv(DER / "balance_data_2026.csv")
print("wide:", bal.shape)
bal.iloc[:3, :6]

wide: (14, 51)


,ID,height,weight,rms_ap HB_t1,rms_ml HB_t1,rms_total HB_t1
0,CA04TU11,1.81,75,0.385116,0.241170,0.544636
1,KA14RE15,1.73,68,0.305889,0.232905,0.432592
2,AN23BE15,1.82,74,0.361183,0.252219,0.510790


In [3]:
tall = bal.melt(id_vars=["ID", "height", "weight"], var_name="col", value_name="value")
parts = tall["col"].str.extract(r"^(?P<param>.+?)\s(?P<condition>HB|LB)_t(?P<block>\d)$")

# A column name that does not match leaves NaN here; nothing should.
assert parts.notna().all().all(), tall.loc[parts.isna().any(axis=1), "col"].unique()

tall = pd.concat([tall.drop(columns="col"), parts], axis=1)
tall["block"] = tall["block"].astype(int)
tall.head(3)

,ID,height,weight,value,param,condition,block
0,CA04TU11,1.81,75,0.385116,rms_ap,HB,1
1,KA14RE15,1.73,68,0.305889,rms_ap,HB,1
2,AN23BE15,1.82,74,0.361183,rms_ap,HB,1


In [4]:
sway = (
    tall.pivot(index=["ID", "height", "weight", "condition", "block"],
               columns="param", values="value")
        .reset_index()
        .rename_axis(columns=None)
)
print("long:", sway.shape, "| expected rows:", bal["ID"].nunique() * len(CONDITIONS) * len(BLOCKS))
sway.head(3)

long: (112, 11) | expected rows: 112


,ID,height,weight,condition,block,PeriodicPower,RemnantPower,VisualWeight,rms_ap,rms_ml,rms_total
0,AN06AN18,1.57,51,HB,1,0.025052,0.170022,0.143514,0.571612,0.611048,0.808381
1,AN06AN18,1.57,51,HB,2,0.029385,0.152912,0.203765,0.592331,0.557023,0.837682
2,AN06AN18,1.57,51,HB,3,0.046196,0.106440,0.193208,0.612553,0.623307,0.866281


## 2. LimeSurvey: keeping only the real participants

The export is the raw survey table, so it also contains pilot runs, tester rows and sessions that were started and abandoned. It is filtered against the participant list from the sway table — that list is the definition of who is in the study — after three codes are repaired: two were typed with an umlaut (`Ü`) where the file names use `U`, and one was entered as the first two letters only.

Both exports are read: the later one should be a superset of the earlier one, and that is checked rather than assumed.

*Выгрузка — сырая таблица опроса, поэтому в ней есть пилотные прогоны, строки тестировщиков и начатые, но брошенные сессии. Фильтрация идёт по списку участников из таблицы раскачивания — именно этот список определяет, кто входит в исследование, — после починки трёх кодов: два набраны с умлаутом (`Ü`) там, где в именах файлов стоит `U`, один введён только первыми двумя буквами.*

*Читаются обе выгрузки: более поздняя должна быть надмножеством ранней, и это проверяется, а не предполагается.*

In [5]:
CODE = "Versuchspersonencode"
sv_new = pd.read_csv(RAW / "limesurvey" / "20260713_results-survey665152.csv", dtype=str)
sv_old = pd.read_csv(RAW / "limesurvey" / "20260702_results-survey665152.csv", dtype=str)

older_ids = set(sv_old[CODE].dropna())
newer_ids = set(sv_new[CODE].dropna())
print("rows: newer", len(sv_new), "| older", len(sv_old))
print("codes only in the older export:", sorted(older_ids - newer_ids) or "none")

rows: newer 31 | older 30
codes only in the older export: none


In [6]:
sv = sv_new.copy()
sv["ID"] = sv[CODE].replace(ID_FIXES)
participants = set(bal["ID"])

print("repaired :", sorted(set(sv_new[CODE].dropna()) & set(ID_FIXES)))
print("discarded:", sorted(set(sv["ID"].dropna()) - participants))

sv = sv[sv["ID"].isin(participants)].copy()
assert sv["ID"].is_unique and set(sv["ID"]) == participants
print("kept     :", len(sv), "rows /", sv["ID"].nunique(), "participants")

repaired : ['AN', 'DI16ÜB31', 'DO11ÜB30']
discarded: ['BA05AI17', 'DO13UE03', 'KA11BE12', 'MA02', 'MA03ON12', 'TE00ST00', 'TR00IAL', 'TR00IARL', 'aa11aa11', 'pilotB']
kept     : 14 rows / 14 participants


## 3. Which session was which condition

Every participant did two sessions, `A` and `B`, one per podcast. The condition label for a session lives in two independent places: the survey field *"Which randomisation order is followed?"*, filled in by the tester, and the VR folder name (`sCA04TU11_B_HB`), written by the recording software.

They are derived separately and compared. A silent disagreement here would attach every boredom rating to the wrong condition, which no downstream check would catch — so the comparison is printed in full rather than reduced to a pass/fail.

*Каждый участник прошёл две сессии, `A` и `B`, по одному подкасту в каждой. Метка условия для сессии лежит в двух независимых местах: поле опроса "Which randomisation order is followed?", заполняемое тестировщиком, и имя папки VR (`sCA04TU11_B_HB`), которое пишет записывающая программа.*

*Они выводятся раздельно и сравниваются. Незамеченное расхождение здесь привязало бы все оценки скуки к неверному условию, и ни одна из последующих проверок это бы не поймала, — поэтому сравнение печатается целиком, а не сводится к «сошлось / не сошлось».*

In [7]:
ORDER_COL = "Which randomisation order is followed? [tick the order of conditions tested]"

# "LB_HB" means session A was LB and session B was HB.
sv["first_condition"] = sv[ORDER_COL].str.split("_").str[0]
order_survey = sv.set_index("ID")["first_condition"]
order_survey.value_counts()

first_condition
LB    7
HB    7
Name: count, dtype: int64

In [8]:
folders = sorted(p.name for p in (RAW / "vr").iterdir() if p.is_dir())
pattern = re.compile(r"^s(?P<ID>[A-Z]{2}\d{2}[A-Z]{2}\d{2})_(?P<session>[AB])_(?P<c>H1?B|LB)")

found = pd.DataFrame(
    [{**m.groupdict(), "folder": f} for f in folders if (m := pattern.match(f))]
)
# "H1B" is a typo in one folder name; the leading H is what identifies the condition.
found["condition"] = np.where(found["c"].str.startswith("H"), "HB", "LB")
found = found.drop(columns="c")
print(len(found), "session folders parsed from", len(folders), "directories")
found.head(3)

28 session folders parsed from 33 directories


,ID,session,folder,condition
0,AN06AN18,A,sAN06AN18_A_LB,LB
1,AN06AN18,B,sAN06AN18_B_HB,HB
2,AN07IE11,A,sAN07IE11_A_HB,HB


In [9]:
# Only participants whose two folders carry two different session letters can define an order.
two_letters = found.groupby("ID")["session"].nunique().eq(2)
order_folders = (found[found["ID"].isin(two_letters[two_letters].index)]
                 .sort_values("session").groupby("ID")["condition"].first())

check = pd.concat([order_survey.rename("from_survey"),
                   order_folders.rename("from_folders")], axis=1)
check["agree"] = check["from_survey"].eq(check["from_folders"])
check

,from_survey,from_folders,agree
ID,,,
CA04TU11,LB,LB,True
KA14RE15,HB,HB,True
AN23BE15,HB,HB,True
EL30AD28,LB,LB,True
BI20OE24,LB,LB,True
CH11RE22,HB,NaN,False
DO11UB30,LB,LB,True
AN07IE11,HB,HB,True
AN06AN18,LB,LB,True


Three participants have both of their folders labelled as session `A`, so the folder names carry no order for them and the comparison is empty rather than wrong. Their condition labels are still present in the folder names and still match; only the *sequence* is unavailable, and the survey field supplies it.

For the remaining participants the two sources agree, so the survey field is used throughout as the single source for the session-to-condition map.

*У трёх участников обе папки помечены как сессия `A`, поэтому имена папок не несут для них порядка — сравнение пустое, а не ошибочное. Метки условий в именах папок при этом есть и совпадают; недоступна только последовательность, и её даёт поле опроса.*

*У остальных источники согласуются, поэтому поле опроса используется как единственный источник карты «сессия → условие».*

In [10]:
def other(c):
    return "LB" if c == "HB" else "HB"

session_map = pd.DataFrame(
    [{"ID": i, "session": s, "condition": first if s == "A" else other(first), "period": p}
     for i, first in order_survey.items()
     for p, s in enumerate(SESSIONS, start=1)]
)
assert len(session_map) == len(participants) * 2
session_map.head(4)

,ID,session,condition,period
0,CA04TU11,A,LB,1
1,CA04TU11,B,HB,2
2,KA14RE15,A,HB,1
3,KA14RE15,B,LB,2


## 4. Boredom rating per block

After each of the four blocks the tester asked *"Langeweile?"* and typed the spoken answer on a 0–10 scale. That is eight columns per participant, named by session and block.

They are melted to long, the session letter and block number are pulled out of the column name, and `99` is recoded to `NaN` — it is the survey's code for a skipped field, and left as a number it would sit far above the scale maximum and drag any mean with it. The result then joins the session map, which turns the session letter into a condition.

*После каждого из четырёх блоков тестировщик спрашивал «Langeweile?» и вносил произнесённый ответ по шкале 0–10. Это восемь колонок на участника, поименованных сессией и блоком.*

*Они разворачиваются в длинный формат, буква сессии и номер блока извлекаются из имени колонки, а `99` перекодируется в `NaN` — это код пропущенного поля в опросе, и оставленный числом он оказался бы много выше максимума шкалы и утянул бы за собой любое среднее. Дальше результат соединяется с картой сессий, которая превращает букву сессии в условие.*

In [11]:
boredom_cols = [f"Check-up questions {s}_{k} [Langeweile?][]" for s in SESSIONS for k in BLOCKS]
assert all(c in sv.columns for c in boredom_cols)

bore = sv[["ID"] + boredom_cols].melt(id_vars="ID", var_name="col", value_name="boredom")
bore = pd.concat(
    [bore.drop(columns="col"),
     bore["col"].str.extract(r"questions (?P<session>[AB])_(?P<block>\d)")], axis=1
)
bore["block"] = bore["block"].astype(int)
bore["boredom"] = pd.to_numeric(bore["boredom"], errors="coerce")
bore["boredom"].value_counts().sort_index()

boredom
0     20
1     20
2     11
3     17
4     10
5      7
6      5
7      8
8      8
9      2
99     4
Name: count, dtype: int64

In [12]:
n_skipped = int(bore["boredom"].eq(MISSING_CODE).sum())
bore.loc[bore["boredom"].eq(MISSING_CODE), "boredom"] = np.nan

bore = bore.merge(session_map, on=["ID", "session"], how="left")
print("ratings:", len(bore), "| recoded from 99:", n_skipped, "| usable:", int(bore["boredom"].notna().sum()))
bore.head(3)

ratings: 112 | recoded from 99: 4 | usable: 108


,ID,boredom,session,block,condition,period
0,CA04TU11,0.0,A,1,LB,1
1,KA14RE15,4.0,A,1,HB,1
2,AN23BE15,5.0,A,1,HB,1


## 5. State boredom before and after each session

The eight-item state boredom scale was answered four times: before and after each of the two sessions. In the export those are four blocks of identically-named columns that pandas has disambiguated with the suffixes `.1`, `.2`, `.3`; their position in the file fixes which is which.

The answers are stored as labels (`"1 - stimme überhaupt nicht zu"`) with only the endpoints spelled out, so the leading integer is extracted before summing. `min_count=8` makes the sum `NaN` when any item is missing instead of silently treating it as a zero.

This is the manipulation check — whether the high-boredom podcast produced more boredom than the low-boredom one. It is not the outcome, and it is not tested here.

*Восьмипунктовая шкала состояния скуки заполнялась четыре раза: до и после каждой из двух сессий. В выгрузке это четыре блока одноимённых колонок, которые pandas развёл суффиксами `.1`, `.2`, `.3`; какой из них какой, определяется положением в файле.*

*Ответы записаны метками (`"1 - stimme überhaupt nicht zu"`), где выписаны только концы шкалы, поэтому перед суммированием извлекается ведущее целое. `min_count=8` делает сумму `NaN`, если пропущен хотя бы один пункт, вместо того чтобы молча счесть его нулём.*

*Это проверка манипуляции — вызвал ли «скучный» подкаст больше скуки, чем «нескучный». Это не исход, и здесь он не тестируется.*

In [13]:
STATE = "im Moment zustimmst"

def state_score(suffix):
    # Sum of the eight state-boredom items for one time point.
    cols = [c for c in sv.columns if STATE in c
            and (c.endswith(suffix) if suffix else not re.search(r"\.\d$", c))]
    assert len(cols) == 8, (suffix, len(cols))
    items = sv[cols].apply(lambda s: pd.to_numeric(s.str.extract(r"^(\d+)")[0], errors="coerce"))
    return items.sum(axis=1, min_count=8)

TIMEPOINTS = {("A", "pre"): "", ("A", "post"): ".1", ("B", "pre"): ".2", ("B", "post"): ".3"}
state = pd.DataFrame({"ID": sv["ID"].values,
                      **{k: state_score(sfx).values for k, sfx in TIMEPOINTS.items()}})
state.head(3)

,ID,"(A, pre)","(A, post)","(B, pre)","(B, post)"
0,CA04TU11,11,10,11,12
1,KA14RE15,24,34,28,28
2,AN23BE15,32,46,45,34


In [14]:
state = state.melt(id_vars="ID", var_name="key", value_name="score")
state[["session", "timepoint"]] = pd.DataFrame(state["key"].tolist(), index=state.index)

state = (state.drop(columns="key")
              .pivot(index=["ID", "session"], columns="timepoint", values="score")
              .reset_index().rename_axis(columns=None)
              .rename(columns={"pre": "state_pre", "post": "state_post"}))
state["state_delta"] = state["state_post"] - state["state_pre"]
state = state.merge(session_map, on=["ID", "session"], how="left")
state.head(4)

,ID,session,state_post,state_pre,state_delta,condition,period
0,AN06AN18,A,12,9,3,LB,1
1,AN06AN18,B,16,12,4,HB,2
2,AN07IE11,A,40,14,26,HB,1
3,AN07IE11,B,21,25,-4,LB,2


## 6. Joining the three tables

The sway table and the boredom table are joined on *(ID, condition, block)*; the session-level state scores are joined on *(ID, condition)* and repeat across that session's four blocks.

Both joins use `how="outer"` with `indicator=True` rather than an inner join. An inner join would drop any key that failed to match and would leave the result looking clean; the indicator makes the unmatched rows visible so they can be counted and, if there are any, explained.

*Таблица раскачивания и таблица скуки соединяются по (ID, условие, блок); оценки состояния уровня сессии — по (ID, условие), и повторяются на все четыре блока этой сессии.*

*Оба соединения делаются через `how="outer"` с `indicator=True`, а не внутренним join. Внутренний выбросил бы любой несовпавший ключ, и результат выглядел бы чистым; индикатор делает несовпавшие строки видимыми — их можно посчитать и, если они есть, объяснить.*

In [15]:
data = sway.merge(
    bore[["ID", "condition", "block", "session", "period", "boredom"]],
    on=["ID", "condition", "block"], how="outer", indicator=True,
)
print(data["_merge"].value_counts().to_string())
data = data.drop(columns="_merge")

_merge
both          112
left_only       0
right_only      0


In [16]:
data = data.merge(
    state[["ID", "condition", "state_pre", "state_post", "state_delta"]],
    on=["ID", "condition"], how="outer", indicator=True,
)
print(data["_merge"].value_counts().to_string())
data = data.drop(columns="_merge")
print("joined:", data.shape)

_merge
both          112
left_only       0
right_only      0
joined: (112, 17)


## 7. Applying the QC decisions

`qc_exclusions.csv` from `01_qc.ipynb` holds three levels of decision and two actions, and the combination determines what happens to a row here.

A **participant**-level `exclude` removes every row for that participant. A **condition**-level `exclude` names one podcast for one participant whose body markers never recorded, and a **trial**-level `exclude` names a single block — both blank the sway parameters for the cells they cover while leaving the boredom rating in place, because the rating is valid and it is the recording that failed. A `keep-and-report` changes no value at all and only sets a flag column, so the participant stays in the analysis and the caveat travels with the data.

The two `exclude` levels are handled by the same code path: a condition-level row simply covers all four blocks, and the `block` column being empty is what says so.

*В `qc_exclusions.csv` из `01_qc.ipynb` три уровня решений и два действия, и их сочетание определяет, что происходит со строкой здесь.*

*`exclude` уровня **участника** убирает все его строки. `exclude` уровня **условия** называет один подкаст одного участника, у которого не записались маркеры тела, а `exclude` уровня **пробы** — один блок; оба обнуляют параметры раскачивания в покрытых ячейках, оставляя оценку скуки на месте, потому что оценка валидна, а отказала запись. `keep-and-report` не меняет ни одного значения и лишь выставляет колонку-флаг: участник остаётся в анализе, а оговорка едет вместе с данными.*

*Оба уровня `exclude` обрабатываются одним кодом: строка уровня условия просто покрывает все четыре блока, и пустая колонка `block` — это и есть признак того, что она их покрывает.*

In [17]:
exc = pd.read_csv(DER / "qc_exclusions.csv")
exc.drop(columns="file")

,level,ID,condition,block,action,reason
0,participant,AN06AN18,NaN,NaN,exclude,body markers flat at zero in every recording; ...
1,condition,DO11UB30,HB,NaN,exclude,body markers flat at zero in every HB recordin...
2,participant,EL30AD28,NaN,NaN,keep-and-report,VR calibration differs by 0.33 m between condi...
3,trial,BI20OE24,HB,1.0,exclude,recording ends at 181.4 s of 210 s; resampling...
4,trial,BI20OE24,LB,1.0,exclude,recording ends at 161.1 s of 210 s; resampling...
5,trial,PE16IN18,LB,1.0,exclude,recording ends at 203.6 s of 210 s; resampling...
6,trial,DO11UB30,LB,4.0,exclude,no recording for this block in the raw data


In [18]:
SWAY_PARAMS = ["PeriodicPower", "RemnantPower", "VisualWeight", "rms_ap", "rms_ml", "rms_total"]

drop_ids = exc.loc[(exc.level == "participant") & (exc.action == "exclude"), "ID"]
flag_ids = exc.loc[exc.action == "keep-and-report", "ID"]

before = len(data)
data = data[~data["ID"].isin(drop_ids)].copy()
data["flag_calibration"] = data["ID"].isin(flag_ids)

print("removed:", list(drop_ids), "->", before - len(data), "rows")
print("flagged:", list(flag_ids), "->", int(data["flag_calibration"].sum()), "rows")

removed: ['AN06AN18'] -> 8 rows
flagged: ['EL30AD28'] -> 8 rows


In [19]:
blanked = exc[(exc.level != "participant") & (exc.action == "exclude")]

# An empty block means the row covers the whole condition.
keys = set()
for r in blanked.itertuples():
    for b in (BLOCKS if pd.isna(r.block) else [int(r.block)]):
        keys.add((r.ID, r.condition, b))

mask = pd.MultiIndex.from_frame(data[["ID", "condition", "block"]]).isin(keys)
data.loc[mask, SWAY_PARAMS] = np.nan
data["no_recording"] = mask

print("rules:", len(blanked), "-> cells blanked:", int(mask.sum()))
data.loc[mask, ["ID", "condition", "block", "PeriodicPower", "boredom"]]

rules: 5 -> cells blanked: 8


,ID,condition,block,PeriodicPower,boredom
24,BI20OE24,HB,1,NaN,6.0
28,BI20OE24,LB,1,NaN,0.0
72,DO11UB30,HB,1,NaN,5.0
73,DO11UB30,HB,2,NaN,6.0
74,DO11UB30,HB,3,NaN,7.0
75,DO11UB30,HB,4,NaN,7.0
79,DO11UB30,LB,4,NaN,1.0
100,PE16IN18,LB,1,NaN,1.0


### Every gap has to have a reason

A missing outcome value that no QC rule explains is the failure mode this whole arrangement exists to prevent: it would be dropped by every test downstream without appearing in any count, and the reported sample size would quietly stop matching the data. So the two are compared directly — the cells where the primary outcome is absent, against the cells the exclusion table covers — and the notebook stops if the first is not contained in the second.

*Пропущенное значение исхода, которое не объясняется ни одним правилом QC, — это ровно тот отказ, ради предотвращения которого всё и построено: оно было бы отброшено каждым тестом ниже по потоку, не попав ни в один счётчик, и заявленный размер выборки тихо перестал бы соответствовать данным. Поэтому две вещи сравниваются напрямую — ячейки, где первичный исход отсутствует, и ячейки, покрытые таблицей исключений, — и ноутбук останавливается, если первое не вложено во второе.*

In [20]:
unexplained = data[data["PeriodicPower"].isna() & ~data["no_recording"]]
print("missing PeriodicPower:", int(data["PeriodicPower"].isna().sum()),
      "| covered by a rule:", int(data["no_recording"].sum()),
      "| unexplained:", len(unexplained))
assert unexplained.empty, unexplained[["ID", "condition", "block"]]

missing PeriodicPower: 8 | covered by a rule: 8 | unexplained: 0


## 8. Writing the analysis table

The columns are ordered so the keys come first and the outcomes last, and the table is written to `data/derived/analysis_long.csv` alongside a codebook naming every column and its origin. From here on nothing reads the survey export or the wide sway table directly.

The counts printed at the end are the ones the Methods section has to state: participants, rows, and how many cells of the primary outcome are actually available.

*Колонки упорядочены так, чтобы ключи шли первыми, а исходы последними; таблица пишется в `data/derived/analysis_long.csv` вместе с кодовой книгой, где названа каждая колонка и её происхождение. Дальше ничто не читает выгрузку опроса или широкую таблицу раскачивания напрямую.*

*Числа, напечатанные в конце, — это те, которые должны быть названы в разделе Methods: участники, строки и сколько ячеек первичного исхода реально доступно.*

In [21]:
KEYS = ["ID", "session", "period", "condition", "block"]
PERSON = ["height", "weight"]
BOREDOM = ["boredom", "state_pre", "state_post", "state_delta"]
FLAGS = ["flag_calibration", "no_recording"]

data = data[KEYS + PERSON + BOREDOM + FLAGS + SWAY_PARAMS].sort_values(KEYS).reset_index(drop=True)
data.head(4)

,ID,session,period,condition,block,height,weight,boredom,state_pre,state_post,state_delta,flag_calibration,no_recording,PeriodicPower,RemnantPower,VisualWeight,rms_ap,rms_ml,rms_total
0,AN07IE11,A,1,HB,1,1.77,68,3.0,14,40,26,False,False,0.016358,0.042411,0.307419,0.381937,0.361750,0.540140
1,AN07IE11,A,1,HB,2,1.77,68,3.0,14,40,26,False,False,0.031991,0.045898,0.367774,0.465818,0.403897,0.658766
2,AN07IE11,A,1,HB,3,1.77,68,4.0,14,40,26,False,False,0.024861,0.072820,0.358578,0.508022,0.350741,0.718451
3,AN07IE11,A,1,HB,4,1.77,68,5.0,14,40,26,False,False,0.022321,0.102399,0.468824,0.608486,0.778177,0.860529


In [22]:
out = DER / "analysis_long.csv"
data.to_csv(out, index=False)

print("participants        :", data["ID"].nunique())
print("rows                :", len(data))
print("PeriodicPower present:", int(data["PeriodicPower"].notna().sum()), "of", len(data))
print("boredom present      :", int(data["boredom"].notna().sum()), "of", len(data))
print("complete both conditions:",
      int(data.dropna(subset=["PeriodicPower"]).groupby("ID")["condition"].nunique().eq(2).sum()))
print("->", out.relative_to(REPO))

participants        : 13
rows                : 104
PeriodicPower present: 96 of 104
boredom present      : 100 of 104
complete both conditions: 12
-> data/derived/analysis_long.csv


In [23]:
CODEBOOK = {
    "ID": "participant code, as written in the VR file names",
    "session": "A or B, the order the two sessions were run in",
    "period": "1 for the first session, 2 for the second",
    "condition": "HB = high-boredom podcast, LB = low-boredom podcast",
    "block": "1-4, the four 210 s balance blocks within a session",
    "height": "body height in m, self-reported (LimeSurvey)",
    "weight": "body weight in kg, self-reported (LimeSurvey)",
    "boredom": "spoken 0-10 boredom rating after this block; 99 recoded to NaN",
    "state_pre": "eight-item state boredom scale, sum, before this session",
    "state_post": "eight-item state boredom scale, sum, after this session",
    "state_delta": "state_post - state_pre",
    "flag_calibration": "participant kept, but VR calibration differs between conditions",
    "no_recording": "no usable recording; sway parameters blanked, rating kept",
    "PeriodicPower": "sway power at the stimulus frequencies (response sway power)",
    "RemnantPower": "sway power not at the stimulus frequencies (random sway power)",
    "VisualWeight": "W from the independent-channel model fit",
    "rms_ap": "RMS anterior-posterior deviation",
    "rms_ml": "RMS medio-lateral deviation",
    "rms_total": "RMS total deviation",
}
assert set(CODEBOOK) == set(data.columns)

book = DER / "analysis_long_codebook.csv"
pd.Series(CODEBOOK, name="description").rename_axis("column").to_csv(book)
print("->", book.relative_to(REPO))

-> data/derived/analysis_long_codebook.csv


## What this notebook did not do

No outcome was compared between conditions, no group was described by its mean, and no test was run. The next step is not `03_analysis.ipynb` but `ANALYSIS_PLAN.md`: the primary outcome, the single confirmatory test, the alpha level and the treatment of the four blocks are written down and committed first. Everything the plan does not name is exploratory and will be reported as such.

*Ни один исход не сравнивался между условиями, ни одна группа не описывалась средним, ни один тест не запускался. Следующий шаг — не `03_analysis.ipynb`, а `ANALYSIS_PLAN.md`: первичный исход, единственный подтверждающий тест, уровень альфа и обращение с четырьмя блоками сначала записываются и коммитятся. Всё, что план не называет, является поисковым и будет так и отмечено.*